# QAOA-GPT inference demo

In [ ]:
from model_interface import QAOA_GPT

In [3]:
import pandas as pd
import numpy as np
import networkx as nx
import random
from tqdm import tqdm
from collections import defaultdict, Counter
import json
from pathlib import Path

## Loading a model

In [4]:
qaoa_gpt_n8_obj = QAOA_GPT(
    model_ckpt='../../ADAPT-GPT/nanoGPT/out-qaoa_n8w_040725_qaoa_mixer_only_ar_0_97/ckpt_6000_gemb__ar_0_96031__er_0_0.pt',
    data_dir='../../ADAPT-GPT/nanoGPT/data/n8w_qaoa_mixer_only_var_ar/qaoa_n8w_040725_qaoa_mixer_only_ar_0_97',
    config_file='../../ADAPT-GPT/nanoGPT/data/n8w_qaoa_mixer_only_var_ar/qaoa_n8w_040725_qaoa_mixer_only_ar_0_97/train_adapt_gpt_config.py',
    temp_folder='temp_data',
    device='cuda'
)

Loading config from: ../../ADAPT-GPT/nanoGPT/data/n8w_qaoa_mixer_only_var_ar/qaoa_n8w_040725_qaoa_mixer_only_ar_0_97/train_adapt_gpt_config.py
Initiating nanoGPT model with padding support
number of parameters: 11.60M


## Generating random graphs

In [5]:
def add_weights_to_nx_graph(nx_graph):
    for u, v in nx_graph.edges():
        cur_weight = round(random.uniform(0, 1), 2)
        while cur_weight == 0:
            cur_weight = round(random.uniform(0, 1), 2)
        nx_graph[u][v]['weight'] = cur_weight
    return nx_graph

In [6]:
tqdm.pandas()

In [7]:
n_graphs = 50
n_nodes = 8

In [8]:
graphs_edgelist_list_dict = dict()

er_graphs_edgelist_list_dict = dict()
for i in range(n_graphs):
    p = random.randrange(6,9) / 10
    cur_graph = nx.erdos_renyi_graph(
        n=n_nodes,
        p=p
    )
    er_graphs_edgelist_list_dict[f'er_graph_{i}'] = add_weights_to_nx_graph(cur_graph)

graphs_edgelist_list_dict.update(er_graphs_edgelist_list_dict)

In [9]:
graphs_edgelist_list_dict['er_graph_2'].edges(data=True)

EdgeDataView([(0, 1, {'weight': 0.59}), (0, 2, {'weight': 0.16}), (0, 3, {'weight': 0.82}), (0, 4, {'weight': 0.86}), (0, 6, {'weight': 0.5}), (0, 7, {'weight': 0.91}), (1, 3, {'weight': 0.27}), (1, 4, {'weight': 0.91}), (1, 5, {'weight': 0.68}), (1, 6, {'weight': 0.64}), (1, 7, {'weight': 0.89}), (2, 3, {'weight': 0.87}), (2, 4, {'weight': 0.94}), (2, 5, {'weight': 0.01}), (2, 6, {'weight': 0.43}), (2, 7, {'weight': 0.62}), (3, 4, {'weight': 0.82}), (3, 5, {'weight': 0.2}), (3, 6, {'weight': 0.92}), (3, 7, {'weight': 0.84}), (4, 5, {'weight': 0.43}), (4, 7, {'weight': 0.52}), (5, 6, {'weight': 0.42}), (5, 7, {'weight': 0.03}), (6, 7, {'weight': 0.55})])

## Generate circuits with QAOA-GPT model

In [10]:
qaoa_gpt_circ_df = qaoa_gpt_n8_obj.generate_circ_from_nx(
    graphs_edgelist_list_dict,
    #calculate_classic_maxcut=False,
    n_samples_per_batch=50, # max number of distinct graphs in a batch
    num_samples=5, # number of samples to draw
    max_new_tokens=150, # number of tokens generated in each sample
    temperature=0.1, # 1.0 = no change, < 1.0 = less random, > 1.0 = more random, in predictions
    top_k=200, # retain only the top_k most likely tokens, clamp others to have 0 probability
)

Preparing graphs...:   0%|                                                                                     | 0/50 [00:00<?, ?it/s]

Restricted license - for non-production use only - expires 2026-11-23


Preparing graphs...: 100%|████████████████████████████████████████████████████████████████████████████| 50/50 [00:05<00:00,  8.50it/s]


Performing FEATHER embedding


Inference. Current batch: n_edges: 13, n_graphs: 1: 100%|█████████████████████████████████████████████| 15/15 [00:07<00:00,  2.05it/s]


In [11]:
qaoa_gpt_circ_df[:3]

,graph,n_edges,q_circuits,adapt_circuit,adapt_full_ar,graph_prefix,energy_gurobi,label,graph_w_jl,graph_weight_norm
0,"[(1, 3), 0.43, (1, 4), 0.73, (1, 6), 0.19, (1,...",20,"[[new_layer_p, 1, -0.52, 0.41, new_layer_p, 1,...",[],None,er_graph_6,-6.43,test_interactive,"[[1, 3, 0.43], [1, 4, 0.73], [1, 6, 0.19], [1,...",1.0
1,"[(1, 4), 0.37, (1, 6), 0.69, (1, 7), 0.92, (2,...",20,"[[new_layer_p, 1, -0.51, 0.38, new_layer_p, 1,...",[],None,er_graph_13,-7.36,test_interactive,"[[1, 4, 0.37], [1, 6, 0.69], [1, 7, 0.92], [2,...",1.0
2,"[(1, 2), 0.67, (1, 3), 0.26, (1, 4), 0.83, (1,...",20,"[[new_layer_p, 1, -0.51, 0.38, new_layer_p, 1,...",[],None,er_graph_16,-6.84,test_interactive,"[[1, 2, 0.67], [1, 3, 0.26], [1, 4, 0.83], [1,...",1.0


## Evaluate circuits

In [12]:
qaoa_gpt_circ_eval_df = qaoa_gpt_n8_obj.eval_circ_df_jl(qaoa_gpt_circ_df)

In [13]:
qaoa_gpt_circ_eval_df[:3]

,graph_prefix,graph,n_edges,q_circuits,adapt_gpt_energies,energy_gurobi
0,er_graph_6,"[[1, 3], 0.43, [1, 4], 0.73, [1, 6], 0.19, [1,...",20,"[[new_layer_p, 1, -0.52, 0.41000000000000003, ...","[-6.079969439354761, -6.094033176939082, -6.12...",-6.43
1,er_graph_13,"[[1, 4], 0.37, [1, 6], 0.6900000000000001, [1,...",20,"[[new_layer_p, 1, -0.51, 0.38, new_layer_p, 1,...","[-7.150357093858568, -7.150936679789527, -7.10...",-7.36
2,er_graph_16,"[[1, 2], 0.67, [1, 3], 0.26, [1, 4], 0.8300000...",20,"[[new_layer_p, 1, -0.51, 0.38, new_layer_p, 1,...","[-6.4410809623356045, -6.555384054557493, -6.5...",-6.84


In [14]:
qaoa_gpt_circ_eval_expl_df = qaoa_gpt_circ_eval_df.explode(['adapt_gpt_energies', 'q_circuits'])

In [16]:
(qaoa_gpt_circ_eval_expl_df['adapt_gpt_energies'] / qaoa_gpt_circ_eval_expl_df['energy_gurobi']).mean()

np.float64(0.965411224793467)

In [17]:
qaoa_gpt_circ_eval_expl_df['q_circuits'].apply(lambda x: x.count('new_layer_p')).mean()

np.float64(7.492)